In [1]:
from datasets import load_dataset

corpus  = load_dataset("BeIR/fiqa", "corpus")["corpus"]     # the haystack (~57k docs)
queries = load_dataset("BeIR/fiqa", "queries")["queries"]   # the questions
qrels   = load_dataset("BeIR/fiqa-qrels") 

c:\Users\nsuh\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(queries)

Dataset({
    features: ['_id', 'title', 'text'],
    num_rows: 6648
})


In [3]:
# datasets are row-indexed; we need id-indexed, so build lookups once
query_text = {row["_id"]: row["text"] for row in queries}
doc_text   = {row["_id"]: row["text"] for row in corpus}

one = qrels['train'][0]
qid = str(one["query-id"])
did = str(one["corpus-id"])


In [4]:
print("QUERY:", query_text[qid])
print("DOC  :", doc_text[did])

QUERY: What is considered a business expense on a business trip?
DOC  : The IRS Guidance pertaining to the subject.  In general the best I can say is your business expense may be deductible.  But it depends on the circumstances and what it is you want to deduct. Travel Taxpayers who travel away from home on business may deduct related   expenses, including the cost of reaching their destination, the cost   of lodging and meals and other ordinary and necessary expenses.   Taxpayers are considered “traveling away from home” if their duties   require them to be away from home substantially longer than an   ordinary day’s work and they need to sleep or rest to meet the demands   of their work. The actual cost of meals and incidental expenses may be   deducted or the taxpayer may use a standard meal allowance and reduced   record keeping requirements. Regardless of the method used, meal   deductions are generally limited to 50 percent as stated earlier.    Only actual costs for lodging may 

In [5]:
from collections import defaultdict

train_pairs = []
for row in qrels['train']:
    qid, did = str(row['query-id']), str(row['corpus-id'])
    if row["score"] > 0:
        train_pairs.append((query_text[qid], doc_text[did]))

relevant = defaultdict(set)
for row in qrels["test"]:
    if row["score"] > 0:
        relevant[str(row["query-id"])].add(str(row["corpus-id"]))

print("train pairs:", len(train_pairs))
print("test queries with answers:", len(relevant))
print("example pair:", train_pairs[3])

train pairs: 14166
test queries with answers: 648
example pair: ('“Business day” and “due date” for bills', "I don't believe Saturday is a business day either. When I deposit a check at a bank's drive-in after 4pm Friday, the receipt tells me it will credit as if I deposited on Monday.  If a business' computer doesn't adjust their billing to have a weekday due date, they are supposed to accept the payment on the next business day, else, as you discovered, a Sunday due date is really the prior Friday. In which case they may be running afoul of the rules that require X number of days from the time they mail a bill to the time it's due.  The flip side to all of this, is to pick and choose your battles in life. Just pay the bill 2 days early. The interest on a few hundred dollars is a few cents per week. You save that by not using a stamp, just charge it on their site on the Friday. Keep in mind, you can be right, but their computer still dings you. So you call and spend your valuable time

In [6]:
import torch
from transformers import AutoModel, AutoTokenizer


In [7]:
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7849.65it/s]


In [8]:
from model import Encoder
enc = Encoder()
v = enc.encode(["what is a 401k?", "how do dividends work?"])
print(v.shape)        # expect (2, 384) for MiniLM
print(v.norm(dim=1))  # expect ~[1.0, 1.0] — confirms normalization worked

True


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1330.99it/s]


torch.Size([2, 384])
tensor([1.0000, 1.0000])


In [9]:
from model import Encoder

enc = Encoder()
print(enc.device)

# two aligned lists, same order — row i of the matrix ↔ corpus_ids[i]
corpus_ids   = list(doc_text.keys())
corpus_texts = [doc_text[i] for i in corpus_ids]

# one batched call — encode() handles the batching internally
corpus_matrix = enc.encode(corpus_texts)   # shape (57k-ish, 384)

print(corpus_matrix.shape)
print(len(corpus_ids), "ids")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2027.84it/s]


cuda
torch.Size([57638, 384])
57638 ids


In [10]:
query_ids = list(query_text.keys())
query_texts = [query_text[i] for i in query_ids]

query_matrix = enc.encode(query_texts)

print(query_matrix.shape)
print(len(query_ids), "ids")

torch.Size([6648, 384])
6648 ids


In [ ]:
score = query_matrix @ corpus_matrix.T
k = 10
top_scores, top_idx = score.topk(k, dim=1)

print(top_idx)

tensor([[51106, 20778, 49423,  ..., 18727, 14267,  1800],
        [51106, 54067, 32732,  ..., 16720, 31397, 51967],
        [50953,  4991, 41129,  ..., 42869, 46217, 56049],
        ...,
        [26562, 23666, 38391,  ..., 14914, 41065, 22531],
        [10965, 39761, 13894,  ..., 14472, 22070, 42814],
        [33136, 54019, 46685,  ...,  6529,  1637, 51136]])


In [2]:
from train import train

model, tokenizer = train(epochs=3, batch_size=8, lr=1e-5)

: 

In [12]:
model.save_pretrained("fiqa-finetuned")
tokenizer.save_pretrained("fiqa-finetuned")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.49it/s]


('fiqa-finetuned\\tokenizer_config.json', 'fiqa-finetuned\\tokenizer.json')

In [13]:
enc = Encoder("fiqa-finetuned")

corpus_ids   = list(doc_text.keys())
corpus_texts = [doc_text[i] for i in corpus_ids]

# one batched call — encode() handles the batching internally
corpus_matrix = enc.encode(corpus_texts)   # shape (57k-ish, 384)

print(corpus_matrix.shape)
print(len(corpus_ids), "ids")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6603.69it/s]


torch.Size([57638, 384])
57638 ids


In [17]:
from data import recall_at_k
score = query_matrix @ corpus_matrix.T
k = 10
top_scores, top_idx = score.topk(k, dim=1)

res = recall_at_k(top_idx, query_ids, corpus_ids, relevant, k=k)
print(res)

0.4074074074074074
